# Valio Aimo – Buffer Margin & Confidence Score Prediction

This notebook builds a unified dataset from:

- `valio_aimo_sales_and_deliveries_junction_2025.csv`
- `valio_aimo_replacement_orders_junction_2025.csv`
- `valio_aimo_purchases_junction_2025.csv`

and trains a predictive model that outputs:

1. **Shortage probability (confidence score)**
2. **Buffer margin prediction**
3. **Uncertainty estimates (50th, 80th, 95th quantiles)**

The goal is to proactively detect upcoming stock-outs before picking and support Valio Aimo’s multimodal AI agent for contacting customers with replacement options.


## 1. Imports and Setup


In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 2. Data Loading and Initial Setup


In [ ]:
print("1. Loading Data Files...")
from pathlib import Path

# Filepaths for the provided CSVs
SALES_FILE = 'valio_aimo_sales_and_deliveries_junction_2025.csv'
REPLACEMENT_FILE = 'valio_aimo_replacement_orders_junction_2025.csv'
PURCHASE_FILE = 'valio_aimo_purchases_junction_2025.csv'
PRODUCT_FILE = 'valio_aimo_product_data_junction_2025.csv'
PRODUCT_RISK_FILE = 'product_risk_scores.csv'
CUSTOMER_IMPACT_FILE = 'customer_impact_scores.csv'
MASTER_FILE = 'master_dataset.csv'

# Load the datasets (be tolerant if some are missing)
def try_read(path, **kwargs):
    try:
        return pd.read_csv(path, **kwargs)
    except FileNotFoundError:
        return None

# Prefer a precomputed master dataset if available
if Path(MASTER_FILE).exists():
    print(f"Loading precomputed master dataset: {MASTER_FILE}")
    master_df = pd.read_csv(MASTER_FILE, low_memory=False)
    # Expose commonly used frames for backward compatibility
    df_sales = master_df.copy()
    df_replacements = try_read(REPLACEMENT_FILE)
    df_purchases = try_read(PURCHASE_FILE)
    df_products = try_read(PRODUCT_FILE)
    df_product_risk = try_read(PRODUCT_RISK_FILE)
    df_customer_impact = try_read(CUSTOMER_IMPACT_FILE)
else:
    df_sales = try_read(SALES_FILE)
    df_replacements = try_read(REPLACEMENT_FILE)
    df_purchases = try_read(PURCHASE_FILE)
    df_products = try_read(PRODUCT_FILE)
    df_product_risk = try_read(PRODUCT_RISK_FILE)
    df_customer_impact = try_read(CUSTOMER_IMPACT_FILE)

if df_sales is None or df_replacements is None or df_purchases is None:
    print("Warning: One or more primary CSVs (sales/replacements/purchases) were not found. Falling back to small placeholder datasets for development.")
    # Minimal placeholder dataframes to allow notebook to run
    df_sales = pd.DataFrame({
        'order_number': [1000, 1000, 1001], 'requested_delivery_date': ['2024-09-02', '2024-09-02', '2024-09-03'],
        'customer_number': [33258, 33258, 31500], 'product_code': [409510, 410914, 406587],
        'order_qty': [5.0, 12.0, 10.0], 'delivered_qty': [5.0, 10.0, 0.0],
        'picking_picked_qty': [5.0, 10.0, 0.0], 'picking_confirmed_time': [203837, 203734, 150000]
    })
    df_replacements = pd.DataFrame({
        'order_number': [3200, 3201, 3202], 'requested_delivery_date': ['2024-09-02', '2024-09-03', '2024-09-03'],
        'customer_number': [33258, 31500, 31500], 'product_code': [409511, 406588, 406589],
        'order_qty': [2.0, 10.0, 5.0]
    })
    df_purchases = pd.DataFrame({
        'order_number': [2300, 2301, 2302], 'po_created_date': ['2024-09-01', '2024-09-01', '2024-09-01'],
        'product_code': [407329, 408060, 407332], 'ordered_qty': [8.0, 3.0, 4.0], 'received_qty': [8, 3, 4]
    })

# Inform about additional optional files
if 'df_products' in globals() and df_products is not None:
    print(f"Loaded product metadata from {PRODUCT_FILE}")
if 'df_product_risk' in globals() and df_product_risk is not None:
    print(f"Loaded product risk scores from {PRODUCT_RISK_FILE}")
if 'df_customer_impact' in globals() and df_customer_impact is not None:
    print(f"Loaded customer impact scores from {CUSTOMER_IMPACT_FILE}")
else:
    print("✓ Primary files loaded (or placeholders used).")


Sales shape: (7357509, 18)
Replacement shape: (15069, 18)
Purchases shape: (782783, 11)


,order_number,order_created_date,order_created_time,requested_delivery_date,customer_number,order_row_number,product_code,order_qty,sales_unit,delivery_number,plant,storage_location,delivered_qty,transfer_number,warehouse_number,picking_confirmed_date,picking_confirmed_time,picking_picked_qty
0,10000000,2024-09-01,336,2024-09-02,33258,10,409510,5.0,ST,20000000.0,30588.0,2012.0,5.0,30000212.0,3001.0,2024-09-01,203837.0,5.0
1,10000000,2024-09-01,336,2024-09-02,33258,40,410914,12.0,ST,20000000.0,30588.0,2012.0,12.0,30000212.0,3001.0,2024-09-01,203734.0,12.0
2,10000000,2024-09-01,336,2024-09-02,33258,50,406587,4.0,ST,20000000.0,30588.0,2012.0,4.0,30000211.0,3001.0,2024-09-01,204149.0,4.0
3,10000000,2024-09-01,336,2024-09-02,33258,60,406588,4.0,ST,20000000.0,30588.0,2012.0,4.0,30000211.0,3001.0,2024-09-01,204124.0,4.0
4,10000000,2024-09-01,336,2024-09-02,33258,70,401369,8.0,BOT,20000000.0,30588.0,2012.0,8.0,30000211.0,3001.0,2024-09-01,205255.0,8.0


## 3. Training Data Creation: Linking Shortage to Replacement

The core challenge is linking the *missing* product (M) to the *replacement* product (R).

**Heuristic**: A replacement order (R) is assumed to cover a shortage (M) if they share the same customer and the requested delivery dates are within a tight window (e.g., 1 day).


In [21]:
def parse_date(df, col):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

# Sales
for col in ["order_created_date", "requested_delivery_date", "picking_confirmed_date"]:
    sales = parse_date(sales, col)

# Replacement
for col in ["order_created_date", "requested_delivery_date", "picking_confirmed_date"]:
    repl = parse_date(repl, col)

# Purchase
for col in ["po_created_date", "requested_delivery_date"]:
    purch = parse_date(purch, col)

sales.dtypes.head(20)


order_number                        int64
order_created_date         datetime64[ns]
order_created_time                  int64
requested_delivery_date    datetime64[ns]
customer_number                     int64
order_row_number                    int64
product_code                        int64
order_qty                         float64
sales_unit                         object
delivery_number                   float64
plant                             float64
storage_location                  float64
delivered_qty                     float64
transfer_number                   float64
warehouse_number                  float64
picking_confirmed_date     datetime64[ns]
picking_confirmed_time            float64
picking_picked_qty                float64
dtype: object

## 4. Feature Engineering from Data Analysis Insights

### A. Temporal Features (from Sales & Purchases Analysis)


In [22]:
sales["date"] = sales["requested_delivery_date"]

sales_agg = (
    sales.groupby(["product_code", "date"], as_index=False)
    .agg(
        total_order_qty=("order_qty", "sum"),
        total_delivered_qty=("delivered_qty", "sum"),
        total_picked_qty=("picking_picked_qty", "sum"),
        n_orders=("order_number", "nunique"),
        n_deliveries=("delivery_number", "nunique"),
    )
)

sales_agg["shortage_amount"] = (
    sales_agg["total_order_qty"] - sales_agg["total_delivered_qty"]
).clip(lower=0)

sales_agg["shortage_flag"] = (sales_agg["shortage_amount"] > 0).astype(int)

sales_agg.head()


,product_code,date,total_order_qty,total_delivered_qty,total_picked_qty,n_orders,n_deliveries,shortage_amount,shortage_flag
0,400001,2025-01-16,1.0,1.0,1.0,1,1,0.0,0
1,400001,2025-03-03,1.0,1.0,1.0,1,1,0.0,0
2,400001,2025-04-02,1.0,1.0,1.0,1,1,0.0,0
3,400001,2025-10-01,1.0,1.0,1.0,1,1,0.0,0
4,400002,2024-09-04,12.0,4.0,4.0,2,1,8.0,1


### B. Risk/Reliability Features (from Purchase & Sales Analysis)

**NOTE**: In a real scenario, these would be loaded from the 'risk_scores.csv' files generated by the other notebooks. We simulate them here.


In [23]:
repl["date"] = repl["requested_delivery_date"]

# Adjust column names if replacement file differs
repl_agg = (
    repl.groupby(["product_code", "date"], as_index=False)
    .agg(
        replacement_order_qty=("order_qty", "sum"),
        replacement_delivered_qty=("delivered_qty", "sum"),
        replacement_picked_qty=("picking_picked_qty", "sum"),
        n_repl_orders=("order_number", "nunique"),
    )
)

repl_agg["replacement_qty"] = (
    repl_agg["replacement_order_qty"] - repl_agg["replacement_delivered_qty"]
).clip(lower=0)

repl_agg["replacement_flag"] = (repl_agg["replacement_qty"] > 0).astype(int)

repl_agg.head()


,product_code,date,replacement_order_qty,replacement_delivered_qty,replacement_picked_qty,n_repl_orders,replacement_qty,replacement_flag
0,400006,2025-06-19,1.0,1.0,1.0,1,0.0,0
1,400006,2025-07-01,1.0,1.0,1.0,1,0.0,0
2,400006,2025-07-14,10.0,10.0,5.0,1,0.0,0
3,400007,2024-12-31,10.0,10.0,10.0,1,0.0,0
4,400008,2025-03-21,2.0,0.0,0.0,1,2.0,1


## 5. Correlation Analysis for Feature Selection

The Target Variable (R_PRODUCT_CODE) is categorical. We cannot calculate standard Pearson correlation directly. We will use a mixed approach:

1. **Feature-Feature Correlation** (to check for multicollinearity)
2. **Feature Importance** via a simple dummy classifier (as proxy for feature-target relationship)


In [24]:
purch["date"] = purch["requested_delivery_date"]

purch_agg = (
    purch.groupby(["product_code", "date"], as_index=False)
    .agg(
        total_po_ordered_qty=("ordered_qty", "sum"),
        total_received_qty=("received_qty", "sum"),
        n_po_rows=("order_number", "nunique"),
    )
)

purch_agg["supply_gap"] = (
    purch_agg["total_po_ordered_qty"] - purch_agg["total_received_qty"]
)

purch_agg.head()


,product_code,date,total_po_ordered_qty,total_received_qty,n_po_rows,supply_gap
0,400001,2025-01-14,1.0,1.0,1,0.0
1,400001,2025-02-27,1.0,1.0,1,0.0
2,400001,2025-04-01,1.0,1.0,1,0.0
3,400001,2025-10-01,1.0,1.0,1,0.0
4,400002,2024-09-04,10.0,10.0,1,0.0


### Interpretation of Correlation


In [25]:
full = sales_agg.merge(
    repl_agg, on=["product_code", "date"], how="outer"
).merge(
    purch_agg, on=["product_code", "date"], how="outer"
)

full.sort_values(["product_code", "date"], inplace=True)
full.reset_index(drop=True, inplace=True)

full.head()


,product_code,date,total_order_qty,total_delivered_qty,total_picked_qty,n_orders,n_deliveries,shortage_amount,shortage_flag,replacement_order_qty,replacement_delivered_qty,replacement_picked_qty,n_repl_orders,replacement_qty,replacement_flag,total_po_ordered_qty,total_received_qty,n_po_rows,supply_gap
0,400001,2025-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0
1,400001,2025-01-16,1.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,400001,2025-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0
3,400001,2025-03-03,1.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,400001,2025-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0


## 6. Feature Importance (Proxy for Feature-Target Relationship)

Use a simple Random Forest Classifier to assess which features are most predictive of the R_PRODUCT_CODE (our target).


In [ ]:
# Fill numeric NaNs with 0
    for col in full.select_dtypes(include=np.number).columns:
        # Preserve missingness by creating an indicator column when NaNs exist
        na_count = int(full[col].isna().sum())
        if na_count > 0:
            full[f"{col}_is_na"] = full[col].isna().astype(int)
            # Fill numeric NaNs with the column median (more robust than 0)
            median = full[col].median()
            if pd.isna(median):
                # As a last resort, fall back to 0 if median is not computable
                median = 0
            full[col] = full[col].fillna(median)
        else:
            # No missing values, leave as-is
            full[col] = full[col]

# Ensure date is valid
full["date"] = pd.to_datetime(full["date"], errors="coerce")

# Target = combined shortage from sales and replacements
full["buffer_target"] = full[["shortage_amount", "replacement_qty"]].max(axis=1)

full[["product_code", "date", "shortage_amount", "replacement_qty", "buffer_target"]].head(10)


,product_code,date,shortage_amount,replacement_qty,buffer_target
0,400001,2025-01-14,0.0,0.0,0.0
1,400001,2025-01-16,0.0,0.0,0.0
2,400001,2025-02-27,0.0,0.0,0.0
3,400001,2025-03-03,0.0,0.0,0.0
4,400001,2025-04-01,0.0,0.0,0.0
5,400001,2025-04-02,0.0,0.0,0.0
6,400001,2025-10-01,0.0,0.0,0.0
7,400002,2024-09-04,8.0,0.0,8.0
8,400002,2024-09-05,0.0,0.0,0.0
9,400002,2024-09-06,0.0,0.0,0.0


In [ ]:
full.sort_values(["product_code", "date"], inplace=True)

def add_rolling(df, col, window, prefix):
    rolled = (
        df.groupby("product_code")[col]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum())
    )
    df[f"{prefix}_roll{window}"] = rolled.fillna(0)  # <- FIXED
    return df



full = add_rolling(full, "total_order_qty", 7,  "demand")
full = add_rolling(full, "total_order_qty", 30, "demand")
full = add_rolling(full, "total_received_qty", 7,  "supply")
full = add_rolling(full, "total_received_qty", 30, "supply")
full = add_rolling(full, "shortage_amount", 7,  "shortage")
full = add_rolling(full, "shortage_amount", 30, "shortage")

full.head()


,product_code,date,total_order_qty,total_delivered_qty,total_picked_qty,n_orders,n_deliveries,shortage_amount,shortage_flag,replacement_order_qty,replacement_delivered_qty,replacement_picked_qty,n_repl_orders,replacement_qty,replacement_flag,total_po_ordered_qty,total_received_qty,n_po_rows,supply_gap,buffer_target,demand_roll7,demand_roll30,supply_roll7,supply_roll30,shortage_roll7,shortage_roll30
0,400001,2025-01-14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,400001,2025-01-16,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,400001,2025-02-27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0
3,400001,2025-03-03,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,0.0,0.0
4,400001,2025-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,0.0


## 7. Model Selection and Architecture Design (Two-Stage Ranking)

Given the vast number of potential replacement products (classes), a single multi-class classifier is inefficient and highly prone to error. We adopt a **two-stage approach**:

### Stage 1: Candidate Generation (Recall Optimization)

**Goal**: Reduce the search space from N products to K (e.g., K=10) candidates.


In [29]:
full = full[full["date"].notnull()].copy()

full["day_of_week"] = full["date"].dt.dayofweek
full["week_of_year"] = full["date"].dt.isocalendar().week.astype(int)
full["month"] = full["date"].dt.month

# Encode product_code as integers
full["product_code_enc"] = full["product_code"].astype("category").cat.codes


### Stage 2: Ranking and Selection (Precision Optimization)

**Goal**: Use all contextual features (customer, time, risk) to select the BEST replacement from the K candidates generated in Stage 1.

LightGBM is chosen for its speed and superior handling of sparse and mixed-type data.


In [30]:
feature_cols = [
    "product_code_enc", "day_of_week", "week_of_year", "month",
    "total_order_qty", "total_delivered_qty", "total_picked_qty",
    "n_orders", "n_deliveries",
    "replacement_order_qty", "replacement_delivered_qty",
    "replacement_picked_qty", "n_repl_orders", "replacement_flag",
    "total_po_ordered_qty", "total_received_qty",
    "n_po_rows", "supply_gap",
    "demand_roll7", "demand_roll30",
    "supply_roll7", "supply_roll30",
    "shortage_roll7", "shortage_roll30",
]

# Keep only columns that exist
feature_cols = [c for c in feature_cols if c in full.columns]

X = full[feature_cols].copy()
y = full["buffer_target"].copy()

X.shape, y.shape


((1475664, 24), (1475664,))

In [31]:
# Train on earliest 80% of timeline
dates_sorted = full["date"].sort_values().unique()
cutoff_index = int(len(dates_sorted) * 0.8)
cutoff_date = dates_sorted[cutoff_index]

train_mask = full["date"] < cutoff_date
test_mask = ~train_mask

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

cutoff_date, X_train.shape, X_test.shape


(Timestamp('2025-08-31 00:00:00'), (1329611, 24), (146053, 24))

## 8. Final System Workflow for the AI Agent


In [32]:
buffer_reg = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
)

buffer_reg.fit(X_train, y_train)

y_pred = buffer_reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


ValueError: Input X contains NaN.
GradientBoostingRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### Example Usage


In [ ]:
# --- Example Usage ---
example_M_product = df_training['M_PRODUCT_CODE'].iloc[0]
example_customer = df_training['customer_number'].iloc[0]

# Simulate real-time stock levels (must be integrated with WMS)
simulated_stock = {
    409511: True, # In stock
    406588: False,
    410914: True,
    409510: True,
    406587: False,
}

print("\n--- Example Prediction for AI Agent ---")
print(f"Missing Product: {example_M_product}, Customer: {example_customer}")
recommendations = predict_replacement(example_M_product, example_customer, simulated_stock)
print("\nAI Agent's Top Proposals (In-Stock):")
for r in recommendations:
    print(f"  - Product {r['R_PRODUCT_CODE']} (Confidence: {r['Confidence_Score']:.2f}, In Stock: {r['In_Stock']})")


# Shortage flag = 1 if buffer >= 1
y_class = (y > 0).astype(int)

y_train_c = y_class[train_mask]
y_test_c  = y_class[test_mask]

clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
)

clf.fit(X_train, y_train_c)

shortage_prob = clf.predict_proba(X_test)[:, 1]

shortage_prob[:10]


In [ ]:
quantiles = [0.5, 0.8, 0.95]
quantile_models = {}
quantile_preds = {}

for q in quantiles:
    qmodel = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
    )
    qmodel.fit(X_train, y_train)
    quantile_models[q] = qmodel
    quantile_preds[q] = qmodel.predict(X_test)


In [ ]:
results = full.loc[test_mask, ["product_code", "date"]].copy()

results["shortage_probability"] = shortage_prob
results["buffer_pred_median"] = quantile_preds[0.5]
results["buffer_pred_80"]     = quantile_preds[0.8]
results["buffer_pred_95"]     = quantile_preds[0.95]

# Operational recommended buffer
results["recommended_buffer"] = results["buffer_pred_80"]

results.head(20)


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, quantile_preds[0.5], alpha=0.3)
plt.xlabel("True buffer")
plt.ylabel("Predicted buffer (median)")
plt.title("True vs Predicted Buffer (Median Quantile)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "--")
plt.show()


In [ ]:
results.to_csv("valio_buffer_predictions_with_confidence.csv", index=False)
results.head()


## Model Summary

The Two-Stage approach ensures high **Recall** (Stage 1: Co-occurrence) and high **Precision** (Stage 2: LightGBM Ranking), providing the AI voice agent with the most relevant and context-aware suggestions.

The final output is the 'good-better-best' list of replacements, ready for the Voice-First interaction.
